# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramithnayak8/ML_pipeline/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because the starter pipeline in this repo already builds most of its
scaffolding: a transparent baseline rule, a client-holdout model, and a ranked queue with reason
codes (`scripts/01`-`05`). Having run that pipeline in notebooks 01-02, I've already seen a
learned model roughly triple the hand-written baseline's Precision@50 on this data — of the four
lanes, this is the one where I have the clearest existing evidence that a model finds something a
rule doesn't. It also has the most direct "who acts on this" story: a content team already reviews
pages weekly (refresh, expand, protect, prune, monitor), and a ranked queue with reason codes
plugs straight into that workflow. I can confirm or swap this lane by the end of Week 4 once I've
looked at the warehouse release.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages across {df['client_id'].nunique()} clients in the starter slice")
print(f"{(df['trend_direction'] == 'down').mean():.1%} of ALL pages are already trending down "
      "-- more than a person can eyeball one at a time")


30,000 pages across 32 clients in the starter slice
54.2% of ALL pages are already trending down -- more than a person can eyeball one at a time


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which pages should a content editor review first for refresh, expansion,
protection, pruning, or monitoring — given that they can only get to a fraction of the inventory
each week?

**Decision this improves:** the order a limited review queue gets worked in, not a one-off
true/false verdict on any single page.

**Who acts, and what they do:** a content strategist / editor works a weekly ranked queue
(shaped like `outputs/refresh_queue.csv`). For each page near the top they pick an action —
refresh, expand, protect (leave it, it's fine), prune, or keep monitoring — using the reason
codes attached to the score, not the score alone.

**Cost of a wrong call:**
- *False positive* (flagged high-priority, wasn't really declining): an editor's scarce hour
  spent on a page that didn't need it. Real cost, but recoverable.
- *False negative* (a genuinely declining page never surfaces near the top): the client keeps
  losing visibility on that page, unnoticed, for another whole review cycle — a cost that
  compounds the longer it's missed.

Because a miss is more expensive than a false alarm, the ranking should lean toward not burying
real decliners, while staying precise enough at the very top that editors keep trusting the list.

In [2]:
visible = df[df["impressions_90d"] >= 500]
declining_visible = (visible["trend_direction"] == "down").sum()

print(f"{len(visible):,} pages get meaningful traffic (impressions_90d >= 500)")
print(f"{declining_visible:,} of those are currently declining")
print("-> far more candidates than a small editorial team reviews in a week by hand;")
print("   a ranked queue -- not a single yes/no flag -- is what makes this list usable.")


16,726 pages get meaningful traffic (impressions_90d >= 500)
9,961 of those are currently declining
-> far more candidates than a small editorial team reviews in a week by hand;
   a ranked queue -- not a single yes/no flag -- is what makes this list usable.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter CSV, in increasing order of how directly they argue for Lane 2:

1. More than half of all 30,000 pages are already trending down (cell above) — too many for
   anyone to review one at a time; a ranked queue is the only practical way to work through it.
2. A crude, fully observable flag (`stale AND visible`) already separates a small high-risk
   group from everyone else by a wide margin — a hint that real signal sits in fields we're
   already allowed to use, before doing anything clever.
3. On the reference pipeline, a random forest beats the hand-written baseline rule by roughly
   3x on Precision@50 — direct evidence that this lane's core promise ("a model beats a rule")
   already holds on this data, not just in theory.

In [3]:
import json

stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
stale_decline_rate = (df.loc[stale_visible, "trend_direction"] == "down").mean()
rest_decline_rate = (df.loc[~stale_visible, "trend_direction"] == "down").mean()

print(f"1) Base rate: {(df['trend_direction'] == 'down').mean():.1%} of all {len(df):,} pages "
      "are already declining")
print(f"2) 'stale + visible' pages ({stale_visible.sum()} of {len(df):,}): "
      f"{stale_decline_rate:.1%} decline rate vs {rest_decline_rate:.1%} for everyone else")

res = json.load(open("../../outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"3) Precision@50 -- baseline rule: {base:.3f}  |  random forest: {rf:.3f}  "
      f"(~{rf / base:.1f}x)")


1) Base rate: 54.2% of all 30,000 pages are already declining
2) 'stale + visible' pages (17 of 30,000): 94.1% decline rate vs 54.2% for everyone else
3) Precision@50 -- baseline rule: 0.240  |  random forest: 0.680  (~2.8x)


## 4. Careful words: what I can and can't claim

**What I can claim, from this notebook:**
- Observed, descriptive statistics from one 30,000-row anonymized snapshot (32 clients).
- A directional relationship between staleness+visibility and decline rate — an association,
  not proof that staleness *causes* decline.
- That a learned ranking model can out-rank a simple hand-written rule *on this snapshot*, under
  client-holdout validation — decision-support evidence, not a guarantee about any one page.

**What I can't claim:**
- That refreshing a page *causes* it to recover — that needs an actual experiment (A/B test or
  similar), not this observational data.
- Anything about a Google ranking-algorithm factor, or about AI search/citation behavior — out
  of scope for this dataset entirely.
- That the "stale + visible" number above is a reliable rule on its own — it comes from a very
  small group in this slice; it's a lead worth testing on the full warehouse, not a finding yet.
- That the random forest's exact Precision@50 is fixed — per this repo's own notes, that number
  moves with library versions; the stable claim is the *relative* lift over the baseline, not
  the third decimal.
- Anything about a specific real client, page, or query — everything here is pseudonymized IDs
  and aggregated numbers, never raw text.

In [4]:
unsafe_markers = ["url", "domain", "keyword", "query", "title", "client_name"]
flagged = [c for c in df.columns if any(m in c.lower() for m in unsafe_markers)]

print("Columns flagged for manual privacy review:", flagged or "none")
print("Sample IDs (should look like pseudonyms, not real names):")
print(df[["content_id", "client_id"]].head(2).to_string(index=False))


Columns flagged for manual privacy review: none
Sample IDs (should look like pseudonyms, not real names):
          content_id         client_id
content_304f48230142 client_f369cb89fc
content_a1fb4e703a9e client_4e07408562


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.